# Retreino e validação USD/BRL — Prophet, SARIMA e promoção offline

**OKR:** comparar uma atualização do modelo sem usar o teste final para escolher a versão. **KPIs:** MAE e acurácia direcional na validação one-step; MAE de campeão, candidato e persistência na janela de promoção.

**Feature engineering:** `Date` é normalizada como índice temporal, `Close` é o alvo, e os recortes são cronológicos; ausências de `Close` saem somente da cópia de preparação. O pipeline produtivo replica essa regra em `src/mlops/`. Snapshot de 975 fechamentos válidos: 70% treino inicial (682), 10% validação (98), 5% lote novo (48), 5% promoção (49) e 10% teste final lacrado (98). O lote e a janela de inferência para apresentação ficam em `data/presentation/`.

O gate compara ambos os modelos **congelados** nas mesmas 49 datas de promoção. O horizonte cresce em cada data; isso avalia a atualização operacional com modelos de idades diferentes, não isola um hiperparâmetro nem é backtesting one-step. O teste final usa o modelo ativo igualmente congelado, mas em janela posterior e, portanto, com horizonte diferente.

Execute agora apenas as células até o gate. Teste final e ajuste corrente de 100% ficam para execução manual futura. Os artefatos legados não são sobrescritos; o contrato do outro checkout ainda precisa ser adaptado aos novos bundles versionados. Resultado experimental, sem recomendação financeira.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
from time import perf_counter
import hashlib
import json
import shutil
import uuid

import pandas as pd
import prophet
from prophet import Prophet
from prophet.serialize import model_to_json, model_from_json

cwd = Path.cwd()
root = cwd.parent if cwd.name == "notebooks" else cwd
raw_path = root / "data/raw/usd_brl_daily.csv"
raw = pd.read_csv(raw_path, parse_dates=["Date"])
if raw.Date.isna().any() or raw.Date.duplicated().any() or not raw.Date.is_monotonic_increasing:
    raise ValueError("Datas ausentes, duplicadas ou desordenadas no CSV bruto")
series = raw[["Date", "Close"]].dropna(subset=["Close"]).rename(columns={"Date": "ds", "Close": "y"}).reset_index(drop=True)
n = len(series)
train_end, validation_end, new_batch_end, promotion_end = (int(n*p) for p in (0.70, 0.80, 0.85, 0.90))
assert (n, train_end, validation_end, new_batch_end, promotion_end) == (975, 682, 780, 828, 877)
if not series.iloc[:validation_end].y.gt(0).all():
    raise ValueError("Preço não positivo na janela de desenvolvimento")
train = series.iloc[:train_end]
validation = series.iloc[train_end:validation_end]
new_batch = series.iloc[validation_end:new_batch_end]
promotion = series.iloc[new_batch_end:promotion_end]
final_test = series.iloc[promotion_end:]  # valores não analisados nesta etapa
assert [len(train), len(validation), len(new_batch), len(promotion), len(final_test)] == [682, 98, 48, 49, 98]
assert train.ds.max() < validation.ds.min() < new_batch.ds.min() < promotion.ds.min() < final_test.ds.min()
split = {
    "train": {"start_index": 0, "end_index_exclusive": train_end, "rows": len(train), "start": str(train.ds.min().date()), "end": str(train.ds.max().date())},
    "validation": {"start_index": train_end, "end_index_exclusive": validation_end, "rows": len(validation), "start": str(validation.ds.min().date()), "end": str(validation.ds.max().date())},
    "new_batch": {"start_index": validation_end, "end_index_exclusive": new_batch_end, "rows": len(new_batch), "start": str(new_batch.ds.min().date()), "end": str(new_batch.ds.max().date())},
    "promotion": {"start_index": new_batch_end, "end_index_exclusive": promotion_end, "rows": len(promotion), "start": str(promotion.ds.min().date()), "end": str(promotion.ds.max().date())},
    "final_test": {"start_index": promotion_end, "end_index_exclusive": n, "rows": len(final_test), "start": str(final_test.ds.min().date()), "end": str(final_test.ds.max().date())},
}
print("CSV bruto:", len(raw), "linhas; válidas:", n, "| removidas só na cópia:", len(raw)-n)
display(pd.DataFrame(split).T)


In [ ]:
CONFIG = {"daily_seasonality": False, "weekly_seasonality": False, "yearly_seasonality": True, "uncertainty_samples": 0}

def direction(value, reference):
    return int(value > reference) - int(value < reference)

def walk_forward_validation(series, start, stop, config):
    started = perf_counter()
    rows = []
    for idx in range(start, stop):
        history = series.iloc[:idx]
        target = series.iloc[idx]
        assert history.ds.max() < target.ds
        model = Prophet(**config).fit(history)
        forecast = float(model.predict(pd.DataFrame({"ds": [target.ds]})).yhat.iloc[0])
        last_close = float(history.y.iloc[-1])
        movements = history.y.diff().dropna()
        majority = 1 if (movements > 0).sum() >= (movements < 0).sum() else -1
        rows.append({
            "cutoff": history.ds.iloc[-1], "target_date": target.ds,
            "last_close": last_close, "actual": float(target.y),
            "prophet_yhat": forecast, "naive_yhat": last_close,
            "actual_direction": direction(target.y, last_close),
            "prophet_direction": direction(forecast, last_close),
            "majority_direction": majority,
        })
    result = pd.DataFrame(rows)
    assert len(result) == stop-start and (result.target_date > result.cutoff).all()
    assert result.target_date.is_monotonic_increasing and result.cutoff.is_monotonic_increasing
    directional = result.loc[result.actual_direction != 0]
    metrics = {
        "n_observations": len(result),
        "n_directional": len(directional),
        "n_stable_actual": int((result.actual_direction == 0).sum()),
        "mae_prophet": float((result.actual-result.prophet_yhat).abs().mean()),
        "mae_naive_last_close": float((result.actual-result.naive_yhat).abs().mean()),
        "direction_accuracy_prophet": float((directional.actual_direction == directional.prophet_direction).mean()) if len(directional) else None,
        "direction_accuracy_majority_history": float((directional.actual_direction == directional.majority_direction).mean()) if len(directional) else None,
        "walk_forward_seconds": round(perf_counter()-started, 2),
    }
    return result, metrics

validation_walk, validation_metrics = walk_forward_validation(series, train_end, validation_end, CONFIG)
print("VALIDAÇÃO one-step — só histórico anterior a cada alvo")
display(pd.Series(validation_metrics))
FROZEN_CONFIG = CONFIG.copy()  # nenhuma mudança após a validação


In [ ]:
# Campeão congelado em 80%; inferência histórica ANTES de incorporar o lote novo.
champion_history = series.iloc[:validation_end].copy()
champion = Prophet(**FROZEN_CONFIG).fit(champion_history)
champion_cutoff = champion_history.ds.iloc[-1]
first_new_batch_date = new_batch.ds.iloc[0]
assert champion_cutoff < first_new_batch_date
historical_yhat = float(champion.predict(pd.DataFrame({"ds": [first_new_batch_date]})).yhat.iloc[0])
champion_last_close = float(champion_history.y.iloc[-1])
historical_direction = "alta" if historical_yhat > champion_last_close else ("queda" if historical_yhat < champion_last_close else "estável")
print({"model": "champion_80_percent", "model_cutoff": str(champion_cutoff.date()),
       "target_date": str(first_new_batch_date.date()), "last_close": champion_last_close,
       "forecast_close": historical_yhat, "direction": historical_direction,
       "context": "inferência histórica antes do retreino; não é cotação atual"})


In [ ]:
# Candidato usa apenas o lote novo e tudo que o precede; nenhum preço de promoção entra no fit.
candidate_history = series.iloc[:new_batch_end].copy()
candidate = Prophet(**FROZEN_CONFIG).fit(candidate_history)
candidate_cutoff = candidate_history.ds.iloc[-1]
assert candidate_cutoff < promotion.ds.min()

promotion_dates = promotion[["ds"]]
champion_forecast = champion.predict(promotion_dates).yhat.to_numpy()
candidate_forecast = candidate.predict(promotion_dates).yhat.to_numpy()
promotion_actual = promotion.y.to_numpy()
# Persistência estática na origem do gate, constante por toda a janela.
gate_last_close = float(candidate_history.y.iloc[-1])
promotion_result = pd.DataFrame({
    "ds": promotion.ds, "actual": promotion_actual,
    "champion_yhat": champion_forecast, "candidate_yhat": candidate_forecast,
    "persistence_yhat": gate_last_close,
})
mae_champion = float((promotion_result.actual-promotion_result.champion_yhat).abs().mean())
mae_candidate = float((promotion_result.actual-promotion_result.candidate_yhat).abs().mean())
mae_persistence = float((promotion_result.actual-promotion_result.persistence_yhat).abs().mean())
promote_candidate = mae_candidate < mae_champion  # empate conserva o campeão
active_name = "candidate_85_percent" if promote_candidate else "champion_80_percent"
active_model = candidate if promote_candidate else champion
active_cutoff = candidate_cutoff if promote_candidate else champion_cutoff
active_rows = len(candidate_history) if promote_candidate else len(champion_history)
gate_metrics = {
    "n_promotion_dates": len(promotion_result),
    "mae_champion": mae_champion, "mae_candidate": mae_candidate,
    "mae_static_persistence": mae_persistence,
    "promoted": promote_candidate, "active_model": active_name,
}
assert len(promotion_result) == 49 and promotion_result.ds.is_monotonic_increasing
assert champion_cutoff < promotion_result.ds.min() and candidate_cutoff < promotion_result.ds.min()
print("PROMOÇÃO OFFLINE — mesmos 49 alvos; dois modelos congelados; horizonte crescente")
display(pd.Series(gate_metrics))

champion_json = model_to_json(champion)
candidate_json = model_to_json(candidate)
champion_sha = hashlib.sha256(champion_json.encode("utf-8")).hexdigest()
candidate_sha = hashlib.sha256(candidate_json.encode("utf-8")).hexdigest()
active_sha = candidate_sha if promote_candidate else champion_sha
gate_metadata = {
    "artifact_scope": "offline_gate_simulation",
    "protocol": "70/10/5/5/10_chronological",
    "source": "Yahoo Finance BRL=X, data/raw/usd_brl_daily.csv",
    "split": split, "validation_metrics": validation_metrics,
    "gate_metrics": gate_metrics, "test_metrics": None,
    "champion": {"cutoff": str(champion_cutoff.date()), "rows": len(champion_history),
                 "last_close": champion_last_close, "sha256": champion_sha},
    "candidate": {"cutoff": str(candidate_cutoff.date()), "rows": len(candidate_history),
                  "last_close": gate_last_close, "sha256": candidate_sha},
    "active_model": active_name, "active_cutoff": str(active_cutoff.date()),
    "active_rows": active_rows, "active_sha256": active_sha,
    "frozen_config": FROZEN_CONFIG, "prophet_version": prophet.__version__,
    "evaluation_note": "modelos congelados, mesmas 49 datas, horizonte crescente; atualização de dados e idade do modelo diferem",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "legacy_artifacts": "não alterados; integração do outro checkout pendente",
}
artifacts = root / "artifacts"
artifacts.mkdir(exist_ok=True)
bundle_dir = artifacts / f"simulation_gate_{promotion.ds.max().date()}_{active_sha[:12]}"
if bundle_dir.exists():
    saved = json.loads((bundle_dir / "metadata.json").read_text(encoding="utf-8"))
    assert saved["champion"]["sha256"] == champion_sha and saved["candidate"]["sha256"] == candidate_sha
    assert hashlib.sha256((bundle_dir / "champion.json").read_bytes()).hexdigest() == champion_sha
    assert hashlib.sha256((bundle_dir / "candidate.json").read_bytes()).hexdigest() == candidate_sha
    print("Bundle imutável existente verificado; nenhum arquivo regravado.")
else:
    tmp_dir = artifacts / f".pending_gate_{uuid.uuid4().hex}"
    tmp_dir.mkdir(exist_ok=False)
    try:
        (tmp_dir / "champion.json").write_text(champion_json, encoding="utf-8")
        (tmp_dir / "candidate.json").write_text(candidate_json, encoding="utf-8")
        (tmp_dir / "metadata.json").write_text(json.dumps(gate_metadata, ensure_ascii=False, indent=2), encoding="utf-8")
        tmp_dir.rename(bundle_dir)  # publicação só depois de todos os arquivos prontos
    except Exception:
        if tmp_dir.exists() and tmp_dir.resolve().parent == artifacts.resolve():
            shutil.rmtree(tmp_dir)
        raise
print("Bundle versionado do gate:", bundle_dir)


## Células manuais futuras

O teste final abaixo permanece lacrado nesta etapa. Quando autorizado, compara o **modelo ativo congelado** nas 98 datas finais com persistência estática na origem do teste. O regime é o mesmo do gate, mas a janela posterior altera o horizonte de cada previsão. O resultado é salvo uma única vez em arquivo exclusivo no bundle do gate; a presença do arquivo impede repetir a avaliação. Não mude a configuração após olhar o teste.

Somente depois ajuste um modelo corrente em 100% do histórico válido. Ele será publicado em **novo** bundle versionado, deixando o legado e o gate intactos. O outro checkout ainda precisará aceitar esse contrato.


In [ ]:
# TESTE FINAL LACRADO — NÃO EXECUTAR NESTA ETAPA.
test_path = bundle_dir / "final_test_metrics.json"
if test_path.exists():
    raise RuntimeError("Teste final já registrado; não repita")
assert active_cutoff < final_test.ds.min()
test_forecast = active_model.predict(final_test[["ds"]]).yhat.to_numpy()
test_last_close = float(series.y.iloc[promotion_end-1])
final_test_metrics = {
    "n_test_dates": len(final_test),
    "mae_active_frozen": float(abs(final_test.y.to_numpy()-test_forecast).mean()),
    "mae_static_persistence": float(abs(final_test.y.to_numpy()-test_last_close).mean()),
    "model_name": active_name, "model_cutoff": str(active_cutoff.date()),
    "model_sha256": active_sha, "frozen_config": FROZEN_CONFIG,
    "test_start": str(final_test.ds.min().date()), "test_end": str(final_test.ds.max().date()),
    "protocol": "modelo congelado; horizonte crescente como no gate; sem tuning após teste",
}
with test_path.open("x", encoding="utf-8") as f:
    json.dump(final_test_metrics, f, ensure_ascii=False, indent=2)
display(pd.Series(final_test_metrics))


In [ ]:
# AJUSTE CORRENTE 100% — NÃO EXECUTAR ANTES DO TESTE FINAL.
test_path = bundle_dir / "final_test_metrics.json"
if not test_path.exists():
    raise RuntimeError("Registre e revise o teste final primeiro")
saved_test_metrics = json.loads(test_path.read_text(encoding="utf-8"))
assert saved_test_metrics["model_sha256"] == active_sha
current_model = Prophet(**FROZEN_CONFIG).fit(series)
current_cutoff = series.ds.iloc[-1]
current_json = model_to_json(current_model)
current_sha = hashlib.sha256(current_json.encode("utf-8")).hexdigest()
current_metadata = dict(gate_metadata)
current_metadata.update({
    "artifact_scope": "current_full_history_100_percent",
    "model_cutoff": str(current_cutoff.date()), "rows_used": len(series),
    "last_close": float(series.y.iloc[-1]),
    "model_sha256": current_sha, "test_metrics": saved_test_metrics,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
})
current_bundle = artifacts / f"simulation_current_{current_cutoff.date()}_{current_sha[:12]}"
if current_bundle.exists():
    raise RuntimeError(f"Bundle imutável já existe: {current_bundle}")
tmp_current = artifacts / f".pending_current_{uuid.uuid4().hex}"
tmp_current.mkdir(exist_ok=False)
try:
    (tmp_current / "model.json").write_text(current_json, encoding="utf-8")
    (tmp_current / "metadata.json").write_text(json.dumps(current_metadata, ensure_ascii=False, indent=2), encoding="utf-8")
    tmp_current.rename(current_bundle)
except Exception:
    if tmp_current.exists() and tmp_current.resolve().parent == artifacts.resolve():
        shutil.rmtree(tmp_current)
    raise
reloaded = model_from_json((current_bundle / "model.json").read_text(encoding="utf-8"))
assert len(reloaded.history) == len(series) and reloaded.history.ds.max() == current_cutoff
print("Novo bundle corrente:", current_bundle)
